<a href="https://colab.research.google.com/github/scudilio/FIAP_MBA-Feature_Engineering/blob/main/Aula4_Features_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 Feature Selection Methods — Hands-on Prático
## FIAP | Feature Engineering | MBA Data Science & AI

**Objetivo:** Aplicar e comparar todas as técnicas de Feature Selection em um dataset real de crédito.

### Técnicas abordadas:
1. **Métodos de Filtro** — Correlação de Pearson, Chi², Informação Mútua, Variance Threshold
2. **Métodos Embarcados** — LASSO (L1), Random Forest Importance, XGBoost Importance
3. **Métodos Wrapper** — Forward Selection, Backward Elimination, RFE (Recursive Feature Elimination)
4. **Comparativo Final** — Qual método selecionou o quê?

**Dataset:** Simulação realista de análise de crédito (aprovação de empréstimos)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Pré-processamento
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Sklearn - Modelos
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# Sklearn - Feature Selection
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, chi2, mutual_info_classif,
    RFE, SequentialFeatureSelector
)

# XGBoost
from xgboost import XGBClassifier

# Config visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


## 📊 1. Carregando o Dataset — Análise de Crédito



In [ ]:
df.to_csv('credit_score_feature_selection.csv', index = False)

In [ ]:
df = pd.read_csv('credit_score_feature_selection.csv')

In [ ]:
df.head()

,renda_mensal,idade,score_credito,tempo_emprego_anos,divida_renda_ratio,num_emprestimos_ativos,historico_atraso,valor_emprestimo,renda_anual,cor_favorita_cod,num_pets,altura_cm,dia_nascimento,codigo_postal,feature_constante,feature_quase_cte,aprovado
0,6621,47,897,0.4,0.040,1,0,3290,79554,9,3,166.0,16,86432,5,0,1
1,4523,48,895,4.0,0.035,1,0,13605,53935,8,1,161.2,14,77925,5,0,1
2,7248,55,694,2.7,0.576,3,0,24531,86888,5,0,164.3,6,25360,5,0,1
3,12256,28,873,10.5,0.356,1,0,5518,147217,1,0,175.7,10,26847,5,0,1
4,4270,52,446,1.1,0.205,1,0,1732,50607,2,2,186.5,17,57565,5,0,1


In [ ]:
# Separar features e target
X = df.drop('aprovado', axis=1)
y = df['aprovado']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Escalar para métodos que precisam
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
print(f"Features totais: {X_train.shape[1]}")

---
## 🔵 2. Métodos de Filtro

### 2.1 Variance Threshold
Remove features com variância abaixo de um limiar. Detecta constantes e quase-constantes.

In [ ]:
# Variance Threshold (limiar = 0.01)
vt = VarianceThreshold(threshold=0.01)
vt.fit(X_train)

variances = pd.Series(vt.variances_, index=X_train.columns).sort_values()

# Features removidas
removed_vt = X_train.columns[~vt.get_support()].tolist()
kept_vt = X_train.columns[vt.get_support()].tolist()

print("🚫 Features REMOVIDAS pelo Variance Threshold:")
for f in removed_vt:
    print(f"   - {f} (variância: {variances[f]:.6f})")
print(f"\n✅ Features mantidas: {len(kept_vt)}")

# Visualização
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#FF5252' if v < 0.01 else '#00BCD4' for v in variances.values]
variances.plot(kind='barh', color=colors, ax=ax)
ax.axvline(x=0.01, color='red', linestyle='--', linewidth=2, label='Threshold = 0.01')
ax.set_xlabel('Variância')
ax.set_title('Variance Threshold — Features por Variância', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 2.2 Correlação de Pearson
Identifica features altamente correlacionadas entre si (redundância) e com o target.

In [ ]:
# Matriz de correlação
corr_matrix = X_train.corr()

# Correlação com o target
corr_with_target = X_train.corrwith(y_train).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap de correlação
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], annot_kws={'size': 7})
axes[0].set_title('Matriz de Correlação (Pearson)', fontsize=13, fontweight='bold')

# Correlação com target
colors_corr = ['#ED1164' if v > 0.1 else '#B0BEC5' for v in corr_with_target.values]
corr_with_target.plot(kind='barh', color=colors_corr, ax=axes[1])
axes[1].axvline(x=0.1, color='red', linestyle='--', alpha=0.7, label='Limiar = 0.1')
axes[1].set_xlabel('|Correlação| com target')
axes[1].set_title('Correlação Absoluta com Target', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# Identificar pares altamente correlacionados (|r| > 0.8)
print("\n🔗 Pares com correlação > 0.8:")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j],
                             round(corr_matrix.iloc[i, j], 3)))
            print(f"   {corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

if not high_corr:
    print("   Nenhum par encontrado acima do limiar.")

### 2.3 Chi-Quadrado (χ²) e Informação Mútua
- **Chi²**: testa independência feature-target (requer dados não-negativos)
- **Informação Mútua**: captura dependência linear E não-linear

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Chi² precisa de dados não-negativos
scaler_mm = MinMaxScaler()
X_train_pos = pd.DataFrame(
    scaler_mm.fit_transform(X_train), columns=X_train.columns
)

# Chi-Quadrado
chi2_scores, chi2_pvalues = chi2(X_train_pos, y_train)
chi2_df = pd.DataFrame({
    'feature': X_train.columns,
    'chi2_score': chi2_scores,
    'p_value': chi2_pvalues
}).sort_values('chi2_score', ascending=False)

# Informação Mútua
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_df = pd.DataFrame({
    'feature': X_train.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chi²
chi2_sorted = chi2_df.sort_values('chi2_score')
colors_chi = ['#ED1164' if s > chi2_sorted['chi2_score'].median() else '#B0BEC5'
              for s in chi2_sorted['chi2_score']]
axes[0].barh(chi2_sorted['feature'], chi2_sorted['chi2_score'], color=colors_chi)
axes[0].set_xlabel('Chi² Score')
axes[0].set_title('Chi-Quadrado (χ²)', fontsize=13, fontweight='bold')

# MI
mi_sorted = mi_df.sort_values('mi_score')
colors_mi = ['#7C4DFF' if s > 0.01 else '#B0BEC5' for s in mi_sorted['mi_score']]
axes[1].barh(mi_sorted['feature'], mi_sorted['mi_score'], color=colors_mi)
axes[1].set_xlabel('Mutual Information Score')
axes[1].set_title('Informação Mútua', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# SelectKBest com MI (top 8)
selector_mi = SelectKBest(score_func=mutual_info_classif, k=8)
selector_mi.fit(X_train, y_train)
selected_mi = X_train.columns[selector_mi.get_support()].tolist()
print(f"\n✅ Top 8 features por Informação Mútua: {selected_mi}")

### 📋 Resumo dos Métodos de Filtro

In [ ]:
# Consolidar resultados dos filtros
filter_results = pd.DataFrame({
    'feature': X_train.columns,
    'variance': vt.variances_,
    'corr_target': X_train.corrwith(y_train).abs().values,
    'chi2_score': chi2_scores,
    'mi_score': mi_scores
})
filter_results['rank_var'] = filter_results['variance'].rank(ascending=False)
filter_results['rank_corr'] = filter_results['corr_target'].rank(ascending=False)
filter_results['rank_chi2'] = filter_results['chi2_score'].rank(ascending=False)
filter_results['rank_mi'] = filter_results['mi_score'].rank(ascending=False)
filter_results['rank_medio'] = filter_results[['rank_var','rank_corr','rank_chi2','rank_mi']].mean(axis=1)

print("🏆 Ranking consolidado dos Métodos de Filtro:")
print(filter_results[['feature','rank_medio']].sort_values('rank_medio').to_string(index=False))

---
## 🟣 3. Métodos Embarcados

### 3.1 LASSO (L1 Regularization)
O LASSO penaliza os coeficientes do modelo, forçando os menos importantes a **zero**.

In [ ]:
# Logistic Regression com L1 (LASSO)
lasso_model = LogisticRegression(
    penalty='l1', solver='saga', C=0.5,
    max_iter=5000, random_state=42
)
lasso_model.fit(X_train_scaled, y_train)

# Coeficientes
lasso_coefs = pd.DataFrame({
    'feature': X_train.columns,
    'coef': lasso_model.coef_[0],
    'abs_coef': np.abs(lasso_model.coef_[0])
}).sort_values('abs_coef', ascending=False)

# Features zeradas pelo LASSO
zero_features = lasso_coefs[lasso_coefs['abs_coef'] < 1e-6]['feature'].tolist()
selected_lasso = lasso_coefs[lasso_coefs['abs_coef'] >= 1e-6]['feature'].tolist()

print(f"✅ Features selecionadas pelo LASSO: {len(selected_lasso)}")
print(f"🚫 Features zeradas: {zero_features}")

# Visualização
fig, ax = plt.subplots(figsize=(12, 6))
lasso_sorted = lasso_coefs.sort_values('coef')
colors_l = ['#7C4DFF' if abs(c) > 1e-6 else '#E0E0E0' for c in lasso_sorted['coef']]
ax.barh(lasso_sorted['feature'], lasso_sorted['coef'], color=colors_l)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente LASSO (L1)')
ax.set_title('LASSO — Coeficientes do Modelo (zeros = features eliminadas)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.2 Random Forest — Feature Importance
Árvores calculam importância pelo ganho de informação (Gini impurity).

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Visualização
fig, ax = plt.subplots(figsize=(12, 6))
rf_sorted = rf_importance.sort_values('importance')
colors_rf = ['#00C853' if v > rf_sorted['importance'].median() else '#B0BEC5'
             for v in rf_sorted['importance']]
ax.barh(rf_sorted['feature'], rf_sorted['importance'], color=colors_rf)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🏆 Top 8 features (RF):")
print(rf_importance.head(8)[['feature', 'importance']].to_string(index=False))

### 3.3 XGBoost — Feature Importance

In [ ]:
# XGBoost
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb_model.fit(X_train, y_train)

xgb_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
xgb_sorted = xgb_importance.sort_values('importance')
colors_xgb = ['#FFB300' if v > xgb_sorted['importance'].median() else '#B0BEC5'
              for v in xgb_sorted['importance']]
ax.barh(xgb_sorted['feature'], xgb_sorted['importance'], color=colors_xgb)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost — Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("🏆 Top 8 features (XGBoost):")
print(xgb_importance.head(8)[['feature', 'importance']].to_string(index=False))

---
## 🟢 4. Métodos Wrapper

### 4.1 RFE — Recursive Feature Elimination
Remove recursivamente a feature menos importante, usando o ranking do modelo.

In [ ]:
# RFE com Decision Tree (rápido e compatível)
estimator = DecisionTreeClassifier(random_state=42, max_depth=5)
rfe = RFE(estimator=estimator, n_features_to_select=8, step=1)
rfe.fit(X_train, y_train)

rfe_results = pd.DataFrame({
    'feature': X_train.columns,
    'selected': rfe.support_,
    'ranking': rfe.ranking_
}).sort_values('ranking')

selected_rfe = rfe_results[rfe_results['selected']]['feature'].tolist()

print(f"✅ Features selecionadas pelo RFE (top 8): {selected_rfe}")
print(f"\n📊 Ranking completo:")
print(rfe_results.to_string(index=False))

### 4.2 Forward Feature Selection (SFS)
Começa sem features e adiciona uma por vez, escolhendo a que mais melhora o score.

In [ ]:
# Sequential Feature Selector - Forward
sfs_forward = SequentialFeatureSelector(
    estimator=DecisionTreeClassifier(random_state=42, max_depth=5),
    n_features_to_select=8,
    direction='forward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
sfs_forward.fit(X_train, y_train)
selected_forward = X_train.columns[sfs_forward.get_support()].tolist()

print(f"✅ Features selecionadas (Forward Selection): {selected_forward}")

### 4.3 Backward Feature Elimination (SBS)
Começa com todas e remove a menos significativa iterativamente.

In [ ]:
# Sequential Feature Selector - Backward
sfs_backward = SequentialFeatureSelector(
    estimator=DecisionTreeClassifier(random_state=42, max_depth=5),
    n_features_to_select=8,
    direction='backward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)
sfs_backward.fit(X_train, y_train)
selected_backward = X_train.columns[sfs_backward.get_support()].tolist()

print(f"✅ Features selecionadas (Backward Elimination): {selected_backward}")

---
## 📊 5. Comparativo Final — Todas as Técnicas
Qual método selecionou quais features? Existe consenso?

In [ ]:
# Consolidar todos os resultados
all_features = X_train.columns.tolist()

# Top 8 de cada método
top8_mi = mi_df.head(8)['feature'].tolist()
top8_rf = rf_importance.head(8)['feature'].tolist()
top8_xgb = xgb_importance.head(8)['feature'].tolist()

comparison = pd.DataFrame({'feature': all_features})
comparison['Variance_OK'] = comparison['feature'].isin(kept_vt).map({True: '✅', False: '❌'})
comparison['MI_Top8'] = comparison['feature'].isin(top8_mi).map({True: '✅', False: '❌'})
comparison['LASSO'] = comparison['feature'].isin(selected_lasso).map({True: '✅', False: '❌'})
comparison['RF_Top8'] = comparison['feature'].isin(top8_rf).map({True: '✅', False: '❌'})
comparison['XGB_Top8'] = comparison['feature'].isin(top8_xgb).map({True: '✅', False: '❌'})
comparison['RFE'] = comparison['feature'].isin(selected_rfe).map({True: '✅', False: '❌'})
comparison['Forward'] = comparison['feature'].isin(selected_forward).map({True: '✅', False: '❌'})
comparison['Backward'] = comparison['feature'].isin(selected_backward).map({True: '✅', False: '❌'})

# Contar votos
method_cols = ['Variance_OK', 'MI_Top8', 'LASSO', 'RF_Top8', 'XGB_Top8', 'RFE', 'Forward', 'Backward']
comparison['votos'] = comparison[method_cols].apply(lambda row: (row == '✅').sum(), axis=1)
comparison = comparison.sort_values('votos', ascending=False)

print("🏆 COMPARATIVO FINAL — Feature Selection")
print("=" * 95)
print(comparison.to_string(index=False))

In [ ]:
# Visualização: Heatmap de consenso
fig, ax = plt.subplots(figsize=(14, 8))

# Preparar dados numéricos para heatmap
heatmap_data = comparison.set_index('feature')[method_cols].replace({'✅': 1, '❌': 0}).astype(int)
heatmap_data = heatmap_data.loc[comparison['feature']]  # manter ordem de votos

sns.heatmap(heatmap_data, annot=True, cmap='RdYlGn', center=0.5,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Selecionada'},
            ax=ax, fmt='d')
ax.set_title('Consenso entre Métodos de Feature Selection', fontsize=15, fontweight='bold')
ax.set_xlabel('Método')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

# Resumo
print("\n📋 RESUMO:")
consensus = comparison[comparison['votos'] >= 6]['feature'].tolist()
print(f"\n🥇 Features com consenso forte (6+ votos): {consensus}")
marginal = comparison[(comparison['votos'] >= 4) & (comparison['votos'] < 6)]['feature'].tolist()
print(f"🥈 Features marginais (4-5 votos): {marginal}")
rejected = comparison[comparison['votos'] <= 2]['feature'].tolist()
print(f"🚫 Features rejeitadas (0-2 votos): {rejected}")

---
## 🧪 6. Validação — Comparar Acurácia

O teste de fogo: usar as features selecionadas por cada método e comparar a acurácia.

In [ ]:
# Baseline: todas as features
results = {}

def evaluate_subset(name, feature_list):
    """Treina RF com subset e retorna accuracy via CV."""
    if not feature_list:
        return 0.0
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    scores = cross_val_score(model, X_train[feature_list], y_train, cv=5, scoring='accuracy')
    results[name] = {'features': len(feature_list), 'accuracy': scores.mean(), 'std': scores.std()}
    return scores.mean()

# Avaliar cada método
evaluate_subset("Todas (baseline)", all_features)
evaluate_subset("Variance Threshold", kept_vt)
evaluate_subset("Mutual Information", top8_mi)
evaluate_subset("LASSO (L1)", selected_lasso)
evaluate_subset("Random Forest Imp.", top8_rf)
evaluate_subset("XGBoost Imp.", top8_xgb)
evaluate_subset("RFE", selected_rfe)
evaluate_subset("Forward Selection", selected_forward)
evaluate_subset("Backward Elim.", selected_backward)
evaluate_subset("Consenso (6+ votos)", consensus)

# Tabela de resultados
results_df = pd.DataFrame(results).T
results_df.columns = ['Nº Features', 'Accuracy (CV)', 'Std']
results_df = results_df.sort_values('Accuracy (CV)', ascending=False)
results_df['Accuracy (CV)'] = results_df['Accuracy (CV)'].round(4)
results_df['Std'] = results_df['Std'].round(4)

print("🏆 COMPARATIVO DE PERFORMANCE")
print("=" * 55)
print(results_df.to_string())

In [ ]:
# Gráfico final
fig, ax = plt.subplots(figsize=(12, 6))

colors_final = ['#ED1164' if name == 'Consenso (6+ votos)' else '#00BCD4'
                for name in results_df.index]

bars = ax.barh(results_df.index, results_df['Accuracy (CV)'], color=colors_final,
               xerr=results_df['Std'], capsize=3, edgecolor='white', linewidth=0.5)

# Annotate
for bar, (name, row) in zip(bars, results_df.iterrows()):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{row['Accuracy (CV)']:.4f} ({int(row['Nº Features'])} feat.)",
            va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Accuracy (5-fold CV)')
ax.set_title('Comparativo de Performance por Método de Feature Selection',
             fontsize=14, fontweight='bold')
ax.set_xlim(left=results_df['Accuracy (CV)'].min() - 0.02)
plt.tight_layout()
plt.show()

print("\n💡 Conclusão: Métodos de Feature Selection permitem reduzir significativamente")
print("   o número de features mantendo (ou melhorando) a performance do modelo!")

---
## 📌 Principais Takeaways

| Método | Tipo | Prós | Contras | Quando usar? |
|--------|------|------|---------|--------------|
| **Variance Threshold** | Filtro | Ultra rápido, remove constantes | Não avalia relação com target | Sempre, como primeiro passo |
| **Correlação** | Filtro | Detecta redundância | Só relações lineares | Datasets com muitas features numéricas |
| **Chi²** | Filtro | Bom para categóricas | Requer dados não-negativos | Features categóricas vs target binário |
| **Informação Mútua** | Filtro | Captura não-linearidade | Pode ser lento em grandes datasets | Quando suspeita de relações não-lineares |
| **LASSO (L1)** | Embarcado | Seleção automática via zeros | Assume linearidade | Modelos lineares e regressão |
| **RF / XGBoost** | Embarcado | Não-linear, robusto | Importância pode variar entre runs | Qualquer problema com árvores |
| **RFE** | Wrapper | Sistemático, usa ranking | Custo computacional | Quando precisa de n features exatas |
| **Forward / Backward** | Wrapper | Avalia combinações | O(n²) ou pior | Poucos features (< 50) |

### 🎯 Recomendação prática:
1. **Comece pelos filtros** (rápidos, eliminam o óbvio)
2. **Refine com embarcados** (LASSO ou tree importance)
3. **Finalize com wrapper** se o dataset for pequeno e precisar do melhor subset
4. **Use votação/consenso** para decisões mais robustas

---
*FIAP — Feature Engineering — MBA Data Science & AI*